In [1]:
import polars as pl
# Скрыть типы данных во всех последующих выводах
#pl.Config.set_tbl_hide_column_data_types(True)

from core.transformations import generalize, suppress_rare_categories, drop_column
from core.privacy_models import KAnonymity

In [4]:
import pandas as pd

In [8]:
# Загрузка данных
df_pd = pd.read_csv("adult.csv")
print("Исходные данные:")
df_pd.head()

Исходные данные:


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [6]:
df_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [9]:
# Загрузка данных
df = pl.read_csv("adult.csv")
print("Исходные данные:")
df.head()

Исходные данные:


age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
i64,str,i64,str,i64,str,str,str,str,str,i64,i64,i64,str,str
25,"""Private""",226802,"""11th""",7,"""Never-married""","""Machine-op-inspct""","""Own-child""","""Black""","""Male""",0,0,40,"""United-States""","""<=50K"""
38,"""Private""",89814,"""HS-grad""",9,"""Married-civ-spouse""","""Farming-fishing""","""Husband""","""White""","""Male""",0,0,50,"""United-States""","""<=50K"""
28,"""Local-gov""",336951,"""Assoc-acdm""",12,"""Married-civ-spouse""","""Protective-serv""","""Husband""","""White""","""Male""",0,0,40,"""United-States""",""">50K"""
44,"""Private""",160323,"""Some-college""",10,"""Married-civ-spouse""","""Machine-op-inspct""","""Husband""","""Black""","""Male""",7688,0,40,"""United-States""",""">50K"""
18,"""?""",103497,"""Some-college""",10,"""Never-married""","""?""","""Own-child""","""White""","""Female""",0,0,30,"""United-States""","""<=50K"""


In [7]:
print(f"Память в мегабайтах: {df.estimated_size(unit='mb')} MB")

Память в мегабайтах: 5.928839683532715 MB


In [30]:
df['native-country'].value_counts().sort('count')

native-country,count
str,u32
"""Holand-Netherlands""",1
"""Hungary""",19
"""Honduras""",20
"""Scotland""",21
"""Outlying-US(Guam-USVI-etc)""",23
…,…
"""Germany""",206
"""Philippines""",295
"""?""",857


In [29]:
[val for val in df['native-country'].value_counts().sort('count')]

[shape: (42,)
 Series: 'native-country' [str]
 [
 	"Holand-Netherlands"
 	"Hungary"
 	"Honduras"
 	"Scotland"
 	"Yugoslavia"
 	…
 	"Germany"
 	"Philippines"
 	"?"
 	"Mexico"
 	"United-States"
 ],
 shape: (42,)
 Series: 'count' [u32]
 [
 	1
 	19
 	20
 	21
 	23
 	…
 	206
 	295
 	857
 	951
 	43832
 ]]

## 